In [1]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch

device = torch.device("cpu")


# dummy dataset 

In [2]:
def create_dummy_compound_graph(num_nodes=25, num_edges=50):
    x = torch.randn(num_nodes, 44)
    edge_index = torch.randint(0, num_nodes, (2, num_edges))
    edge_attr = torch.randn(num_edges, 10)
    lap_pos_enc = torch.randn(num_nodes, 8)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)


def create_dummy_protein_graph(num_nodes=150, num_edges=300):
    x = torch.randn(num_nodes, 41)
    edge_index = torch.randint(0, num_nodes, (2, num_edges))
    edge_attr = torch.randn(num_edges, 5)
    lap_pos_enc = torch.randn(num_nodes, 8)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)


def create_dummy_protein_embedding(num_residues=150):
    return np.random.randn(num_residues, 320).astype(np.float32)


def create_dummy_dataset(num_samples=20):
    dataset = []
    for _ in range(num_samples):
        compound_nodes = np.random.randint(15, 40)
        compound_edges = compound_nodes * 2
        protein_nodes = np.random.randint(100, 200)
        protein_edges = protein_nodes * 2
        compound_graph = create_dummy_compound_graph(compound_nodes, compound_edges)
        protein_graph = create_dummy_protein_graph(protein_nodes, protein_edges)
        protein_embedding = create_dummy_protein_embedding(protein_nodes)
        label = np.random.uniform(4.0, 10.0)
        dataset.append({
            'compound_graph': compound_graph,
            'protein_graph': protein_graph,
            'protein_embedding': protein_embedding,
            'label': label
        })
    return dataset


class DTIDatasetPyG(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        return item['compound_graph'], item['protein_graph'], item['protein_embedding'], item['label']


def custom_collate_pyg(batch):  
    compound_graphs = [item[0] for item in batch]
    protein_graphs = [item[1] for item in batch]
    protein_embeddings = [item[2] for item in batch]
    labels = [item[3] for item in batch]

    compound_batch = Batch.from_data_list(compound_graphs)
    protein_batch = Batch.from_data_list(protein_graphs)

    max_protein_len = max(emb.shape[0] for emb in protein_embeddings)
    padded_embeddings = []
    for emb in protein_embeddings:
        if emb.shape[0] < max_protein_len:
            pad_size = max_protein_len - emb.shape[0]
            padded = np.pad(emb, ((0, pad_size), (0, 0)), mode='constant', constant_values=0)
            padded_embeddings.append(padded)
        else:
            padded_embeddings.append(emb)

    protein_embedding_batch = torch.FloatTensor(np.array(padded_embeddings))
    label_batch = torch.FloatTensor(labels)
    return compound_batch, protein_batch, protein_embedding_batch, label_batch


def demonstrate_shapes():
    dummy_data = create_dummy_dataset(num_samples=20)
    print(f"Created {len(dummy_data)} samples. Sample 0 embedding shape: {dummy_data[0]['protein_embedding'].shape}")

    dataset = DTIDatasetPyG(dummy_data)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_pyg, drop_last=True)
    for batch_idx, data in enumerate(dataloader):
        compound_graph, protein_graph, protein_embedding, label = data
        print(f"Batch {batch_idx + 1}: compound_nodes={compound_graph.x.shape[0]}, protein_nodes={protein_graph.x.shape[0]}, embedding_shape={protein_embedding.shape}, labels={label.tolist()}")
        print(compound_graph.edge_index)
        break



In [3]:
demonstrate_shapes()

Created 20 samples. Sample 0 embedding shape: (116, 320)
Batch 1: compound_nodes=107, protein_nodes=540, embedding_shape=torch.Size([4, 150, 320]), labels=[7.988793849945068, 8.960445404052734, 9.400137901306152, 8.21613883972168]
tensor([[ 15,   1,   5,  10,   3,   9,   7,   5,  11,  14,  14,  12,  14,   3,
          10,  10,   1,  10,   0,   8,   4,   4,   4,   8,   9,   9,  16,  11,
           9,  13,  14,   1,  12,  16,  10,   9,  20,  38,  38,  43,  39,  34,
          20,  39,  33,  39,  27,  23,  36,  31,  42,  30,  37,  42,  42,  47,
          34,  40,  40,  38,  46,  22,  23,  24,  30,  39,  29,  20,  22,  26,
          43,  47,  47,  27,  30,  42,  35,  45,  23,  47,  33,  43,  31,  36,
          29,  21,  36,  27,  26,  44,  42,  39,  22,  18,  29,  41,  36,  38,
          67,  54,  51,  62,  68,  54,  55,  62,  59,  65,  53,  62,  59,  62,
          54,  66,  64,  50,  58,  60,  64,  52,  53,  58,  49,  52,  68,  53,
          63,  50,  55,  52,  61,  68,  52,  61,  68,  67,

# graph_transformer_edge_layer

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter

from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

class MultiHeadAttentionLayer(MessagePassing):
    def __init__(self, in_dim, out_dim, num_heads, use_bias):
        super().__init__(aggr='add', node_dim=0)
        
        self.out_dim = out_dim
        self.num_heads = num_heads
        
        if use_bias:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=True)
        else:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=False)
    
    def forward(self, h, edge_index, e):
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        
        # Project to Q, K, V
        Q_h = self.Q(h).view(num_nodes, self.num_heads, self.out_dim)
        K_h = self.K(h).view(num_nodes, self.num_heads, self.out_dim)
        V_h = self.V(h).view(num_nodes, self.num_heads, self.out_dim)
        proj_e = self.proj_e(e).view(num_edges, self.num_heads, self.out_dim)
        
        # Propagate messages
        h_out, e_out = self.propagate(
            edge_index, 
            Q=Q_h, 
            K=K_h, 
            V=V_h, 
            proj_e=proj_e,
            num_nodes=num_nodes
        )
        
        return h_out, e_out
    
    def message(self, K_j, Q_i, V_j, proj_e, index, num_nodes):
        # K_j: [num_edges, num_heads, out_dim] - source node K
        # Q_i: [num_edges, num_heads, out_dim] - target node Q
        # V_j: [num_edges, num_heads, out_dim] - source node V
        # proj_e: [num_edges, num_heads, out_dim] - edge features
        # index: target node indices for each edge
        
        # Compute attention scores: Q·K / sqrt(d_k)
        score = (Q_i * K_j) / (self.out_dim ** 0.5)  # [num_edges, num_heads, out_dim]
        
        # Modulate by edge features (element-wise multiplication)
        score = score * proj_e  # [num_edges, num_heads, out_dim]
        
        # Store edge output (before softmax, as per DGL implementation)
        self.e_out = score
        
        # Sum across feature dimension for softmax
        score_scalar = score.sum(dim=-1)  # [num_edges, num_heads]
        
        # Clamp for numerical stability
        score_scalar = score_scalar.clamp(-5, 5)
        
        # Apply softmax per target node
        attn = softmax(score_scalar, index, num_nodes=num_nodes)  # [num_edges, num_heads]
        
        # Expand for multiplication with V
        attn = attn.unsqueeze(-1)  # [num_edges, num_heads, 1]
        
        # Weight the values
        msg = V_j * attn  # [num_edges, num_heads, out_dim]
        
        return msg
    
    def aggregate(self, inputs, index, num_nodes):
        # inputs: [num_edges, num_heads, out_dim]
        # Sum messages going to each node
        h_out = scatter(inputs, index, dim=0, dim_size=num_nodes, reduce='sum')
        return h_out
    
    def update(self, aggr_out):
        # aggr_out: [num_nodes, num_heads, out_dim]
        return aggr_out, self.e_out
    



class GraphTransformerLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, dropout, layer_norm=True, batch_norm=False, residual=True, use_bias=False):
        super().__init__()
        self.in_channels = in_dim
        self.out_channels = out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.residual = residual
        self.layer_norm = layer_norm
        self.batch_norm = batch_norm
        
        self.attention = MultiHeadAttentionLayer(in_dim, out_dim // num_heads, num_heads, use_bias)
        
        self.O_h = nn.Linear(out_dim, out_dim)
        self.O_e = nn.Linear(out_dim, out_dim)
        
        if self.layer_norm:
            self.layer_norm1_h = nn.LayerNorm(out_dim)
            self.layer_norm1_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm1_h = nn.BatchNorm1d(out_dim)
            self.batch_norm1_e = nn.BatchNorm1d(out_dim)
        
        self.FFN_h_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_h_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        self.FFN_e_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_e_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        if self.layer_norm:
            self.layer_norm2_h = nn.LayerNorm(out_dim)
            self.layer_norm2_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm2_h = nn.BatchNorm1d(out_dim)
            self.batch_norm2_e = nn.BatchNorm1d(out_dim)
    
    def forward(self, h, edge_index, e):
        h_in1 = h
        e_in1 = e
      
        h_attn_out, e_attn_out = self.attention(h, edge_index, e)
        
        h = h_attn_out.view(-1, self.out_channels)
        e = e_attn_out.view(-1, self.out_channels)
        
        h = F.dropout(h, self.dropout, training=self.training)
        e = F.dropout(e, self.dropout, training=self.training)
        
        h = self.O_h(h)
        e = self.O_e(e)
        
        if self.residual:
            h = h_in1 + h
            e = e_in1 + e
        
        if self.layer_norm:
            h = self.layer_norm1_h(h)
            e = self.layer_norm1_e(e)
        
        if self.batch_norm:
            h = self.batch_norm1_h(h)
            e = self.batch_norm1_e(e)
        
        h_in2 = h
        e_in2 = e
        
        h = self.FFN_h_layer1(h)
        h = F.relu(h)
        h = F.dropout(h, self.dropout, training=self.training)
        h = self.FFN_h_layer2(h)
        
        e = self.FFN_e_layer1(e)
        e = F.relu(e)
        e = F.dropout(e, self.dropout, training=self.training)
        e = self.FFN_e_layer2(e)
        
        if self.residual:
            h = h_in2 + h
            e = e_in2 + e
        
        if self.layer_norm:
            h = self.layer_norm2_h(h)
            e = self.layer_norm2_e(e)
        
        if self.batch_norm:
            h = self.batch_norm2_h(h)
            e = self.batch_norm2_e(e)
        
        return h, e


class GraphTransformerCompound(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        
        data = data.to(self.device)
        h = data.x.float().to(self.device)
        h_lap_pos_enc = data.lap_pos_enc.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h


class GraphTransformerProtein(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        data = data.to(self.device)
        h = data.x.float().to(self.device)
        h_lap_pos_enc = data.lap_pos_enc.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h

In [5]:
# Model initialization
compound_model = GraphTransformerCompound(
    device=device,
    n_layers=3,
    node_dim=44,
    edge_dim=10,
    hidden_dim=128,
    out_dim=128,
    n_heads=8,
    in_feat_dropout=0.1,
    dropout=0.1,
    pos_enc_dim=8
).to(device)

protein_model = GraphTransformerProtein(
    device=device,
    n_layers=3,
    node_dim=41,
    edge_dim=5,
    hidden_dim=128,
    out_dim=128,
    n_heads=8,
    in_feat_dropout=0.1,
    dropout=0.1,
    pos_enc_dim=8
).to(device)

# Create dummy dataset
dummy_data = create_dummy_dataset(num_samples=4)
dataset = DTIDatasetPyG(dummy_data)

# Create DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=custom_collate_pyg, 
    drop_last=True
)

# Get batch from DataLoader
for compound_batch, protein_batch, protein_embedding, label in dataloader:
    # Move to device
    compound_batch = compound_batch.to(device)
    protein_batch = protein_batch.to(device)
    protein_embedding = protein_embedding.to(device)
    label = label.to(device)
    
    # Forward pass
    with torch.no_grad():
        compound_node_features = compound_model(compound_batch)
        protein_node_features = protein_model(protein_batch)
    
    # Print shapes
    print("Compound output:", compound_node_features.shape)
    print("Protein output:", protein_node_features.shape)
    print("Protein embedding:", protein_embedding.shape)
    print("Labels:", label.shape)
    
    break  # Only process first batch

Compound output: torch.Size([138, 128])
Protein output: torch.Size([590, 128])
Protein embedding: torch.Size([4, 163, 320])
Labels: torch.Size([4])
